In [1]:
import pandas as pd
import geopandas as gpd

from utils.rsei_utils import (
    load_tabular_water_microdata,
    get_source_chemicals, get_source_facilities
)

In [2]:
datadir = '/projects/standard/lenkne/oboiko/EJ/data/'
aoi = gpd.read_file(datadir + 'aoi_huc12_boundaries.gpkg')
years = range(2008,2023)
universe = 'core01'

ERROR 1: PROJ: proj_create_from_database: Open of /users/2/oboiko/.conda/envs/geo/share/proj failed


In [3]:
%%time
# pre-load tabular Geographic Microdata
tabular_gm_on = load_tabular_water_microdata(site="Onsite", universe=universe)
tabular_gm_off = load_tabular_water_microdata(site="Offsite", universe=universe)

CPU times: user 1min 36s, sys: 5.44 s, total: 1min 41s
Wall time: 1min 47s


In [4]:
# get facilities from on- and off-site releases
facilities_on = get_source_facilities("Onsite", tabular_gm_on, aoi=aoi, universe=universe, years=years)
facilities_off = get_source_facilities("Offsite", tabular_gm_off, aoi=aoi, universe=universe, years=years)

Got 673 unique facilities for the study period
Got 800 unique facilities for the study period


In [5]:
cols = [
    "FacilityID", "FacilityName", "Latitude", "Longitude", 
    "Street", "City", "County", "State", "ZIPCode",
    "2022NAICSCode", "TRIIndustrySector", "LongName"
]
facilities_on['site'] = 'on'
facilities_off['site'] = 'off'
# Concatenate
facilities_all = pd.concat([facilities_on, facilities_off])
# Define the aggregation logic
# Replace 'facility_id' with the actual column name you use to identify facilities
aggregation_rules = {
    'ToxConc': 'sum',                                        # Sum the concentrations
    'site': lambda x: 'both' if x.nunique() > 1 else x.iloc[0] # Set 'both' if in both dfs
}
# Group by facility and apply the rules
facilities_all = facilities_all.groupby(cols).agg(aggregation_rules).reset_index().sort_values(by="ToxConc", ascending=False)
facilities_all['share ToxConc'] = facilities_all['ToxConc']/facilities_all['ToxConc'].sum()*100
print(len(facilities_all), facilities_all.head(10)['share ToxConc'].sum())
facilities_all.head(10)

1278 79.45241903260327


,FacilityID,FacilityName,Latitude,Longitude,Street,City,County,State,ZIPCode,2022NAICSCode,TRIIndustrySector,LongName,ToxConc,site,share ToxConc
395,52732DMCRN1251B,ADM CORN PROCESSING,41.819600,-90.212200,1251 BEAVER CHANNEL PKWY,CLINTON,CLINTON,IA,52732,311221,311 Food,Wet Corn Milling and Starch Manufacturing,1.206504e+06,both,18.070507
1149,70805XXNCH4999S,EXXONMOBIL BATON ROUGE CHEMICAL PLANT (PART),30.495769,-91.173111,4999 SCENIC HWY,BATON ROUGE,EAST BATON ROUGE PARISH,LA,70805,325199,325 Chemicals,All Other Basic Organic Chemical Manufacturing,7.432161e+05,on,11.131580
179,3912WBRGHT21MRA,BRIGHTON TRU EDGE,31.529770,-91.434760,21 MORAN RD,NATCHEZ,ADAMS,MS,39120,332420,332 Fabricated Metals,Metal Tank (Heavy Gauge) Manufacturing,6.644384e+05,on,9.951681
1116,70734NRYLCPOBOX,LION COPOLYMER GEISMAR LLC,30.205046,-91.005453,36191 LOUISIANA HWY 30,GEISMAR,ASCENSION PARISH,LA,70734,325212,325 Chemicals,Synthetic Rubber Manufacturing,6.038986e+05,on,9.044940
330,52156SWSSVHWY18,PRAIRIE FARMS,43.055850,-91.443560,11744 EDGEWOOD AVE,LUANA,CLAYTON,IA,52156,311513,311 Food,Cheese Manufacturing,4.739131e+05,on,7.098072
1141,70791GRGPCZACHA,GEORGIA-PACIFIC CONSUMER OPERATIONS LLC,30.650644,-91.281167,1000 W MOUNT PLEASANT RD,ZACHARY,EAST BATON ROUGE PARISH,LA,70791,322121,322 Paper,Paper (except Newsprint) Mills,3.690672e+05,on,5.527735
1000,63461MRCNCSTATE,BASF CORP - HANNIBAL SITE,39.834118,-91.436791,3150 HWY JJ,PALMYRA,MARION,MO,63461,325320,325 Chemicals,Pesticide and Other Agricultural Chemical Manu...,3.545490e+05,on,5.310286
843,62040GRNTC20THS,U.S. STEEL GRANITE CITY WORKS,38.695400,-90.136700,1951 STATE ST,GRANITE CITY,MADISON,IL,62040,331110,331 Primary Metals,Iron and Steel Mills and Ferroalloy Manufactur...,3.407917e+05,on,5.104235
36,38053SMSWD3450F,KOPPERS PERFORMANCE CHEMCIALS,35.273500,-89.947920,3450 FITE RD,MILLINGTON,SHELBY,TN,38053,325320,325 Chemicals,Pesticide and Other Agricultural Chemical Manu...,3.132219e+05,on,4.691307
599,55343SMNCS5951C,SUEZ WTS SOLUTIONS USA INC.,44.894920,-93.440440,5951 CLEARWATER DR,MINNETONKA,HENNEPIN,MN,55343,333998,333 Machinery,All Other Miscellaneous General Purpose Machin...,2.351565e+05,off,3.522075


In [4]:
chemicals_on = get_source_chemicals("Onsite", tabular_gm_on, aoi=aoi, universe=universe, years=years)
chemicals_off = get_source_chemicals("Offsite", tabular_gm_off, aoi=aoi, universe=universe, years=years)

Got 186 unique chemicals for the study period
Got 121 unique chemicals for the study period


In [17]:
chemicals_all = pd.concat([chemicals_on, chemicals_off])
chemicals_all = chemicals_all.drop('ChemicalNumber', axis=1)
aggregation_rules = {
    'Chemical': lambda x: ', '.join(x.unique()),
    'ToxConc': 'sum'
}
# using MetalCombinedChemNum to combine metals and their associated compounds into one
chemicals_all = chemicals_all.groupby(['MetalCombinedChemNum']).agg(aggregation_rules).reset_index()
chemicals_all = chemicals_all.sort_values(by="ToxConc", ascending=False)
chemicals_all['share ToxConc'] = chemicals_all['ToxConc']/chemicals_all['ToxConc'].sum()*100
print(len(chemicals_all), chemicals_all.head(10)['share ToxConc'].sum())
chemicals_all.head(10)

193 93.39167030416277


,MetalCombinedChemNum,Chemical,ToxConc,share ToxConc
179,582,"Vanadium compounds, Vanadium (except when cont...",1.593312e+06,23.863967
0,3,Acetaldehyde,1.340016e+06,20.070199
52,152,"Chromium, Chromium compounds (except for chrom...",1.169533e+06,17.516773
16,39,"Arsenic compounds, Arsenic",5.522414e+05,8.271241
133,410,Nitric acid,4.655914e+05,6.973434
111,346,"Lead compounds, Lead",3.051864e+05,4.570955
106,329,Hydrazine,2.891963e+05,4.331461
86,273,"1,4-Dioxane",2.591397e+05,3.881287
188,609,Polycyclic aromatic compounds,1.510179e+05,2.261882
169,545,Thiourea,1.101962e+05,1.650472


In [18]:
#facilities_all.to_csv(datadir + 'source_facilities.csv')
#chemicals_all.to_csv(datadir + 'source_chemicals.csv')